In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, BooleanType
import pandas as pd
import numpy as np
import sys
from pathlib import Path
import tempfile
import os

from hypex.dataset import Dataset, TargetRole, TreatmentRole
from hypex.dataset.backends import SparkDataset

# Создаем Spark сессию
sp_s = (SparkSession.builder 
    .appName("HypEx-Spark-Tests") 
    .master("local[*]") 
    .config("spark.driver.memory", "4g") 
    .config("spark.sql.shuffle.partitions", "4") 
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

sp_s.sparkContext.setLogLevel("WARN")
print(f"Spark Version: {sp_s.version}")

26/03/02 19:46:06 WARN Utils: Your hostname, MacBook-Pro-Danil.local resolves to a loopback address: 127.0.0.1; using 10.240.114.202 instead (on interface en0)
26/03/02 19:46:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/02 19:46:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Version: 3.5.8


In [2]:
# === СОЗДАНИЕ ТЕСТОВЫХ ДАННЫХ ===
print("=" * 60)
print("СОЗДАНИЕ ТЕСТОВЫХ ДАННЫХ")
print("=" * 60)

# Числовые данные для арифметических операций
test_numeric_data = {
    'id': [1, 2, 3, 4, 5],
    'value1': [10.5, 20.3, 30.1, 40.7, 50.9],
    'value2': [5, 10, 15, 20, 25],
    'value3': [100, 200, 300, 400, 500]
}
test_numeric_df = pd.DataFrame(test_numeric_data)

# Смешанные данные для навигации
test_mixed_data = {
    'id': [1, 2, 3, 4, 5],
    'name': ['A', 'B', 'C', 'D', 'E'],
    'value': [10.5, 20.3, 30.1, 40.7, 50.9],
    'category': ['X', 'Y', 'X', 'Y', 'Z'],
    'active': [True, False, True, True, False]
}
test_mixed_df = pd.DataFrame(test_mixed_data)

# Создаем датасеты
ds_numeric = SparkDataset(data=test_numeric_df, session=sp_s)
ds_mixed = SparkDataset(data=test_mixed_df, session=sp_s)

print(f"✅ Числовой датасет: {ds_numeric.shape}")
print(f"✅ Смешанный датасет: {ds_mixed.shape}")
print(f"✅ Колонки числового: {ds_numeric.columns}")
print(f"✅ Колонки смешанного: {ds_mixed.columns}")

СОЗДАНИЕ ТЕСТОВЫХ ДАННЫХ


✅ Числовой датасет: (5, 4)
✅ Смешанный датасет: (5, 5)
✅ Колонки числового: ['id', 'value1', 'value2', 'value3']
✅ Колонки смешанного: ['id', 'name', 'value', 'category', 'active']


In [3]:
# === ТЕСТ: Инициализация SparkDataset ===
print("=" * 60)
print("ТЕСТ: Инициализация SparkDataset")
print("=" * 60)

test_results = []

try:
    # Из pandas DataFrame
    ds = SparkDataset(data=test_numeric_df, session=sp_s)
    assert ds.data is not None, "Data не инициализирован"
    assert ds.shape[0] == 5, f"Ожидалось 5 строк, получено {ds.shape[0]}"
    test_results.append(("✅ PASSED", "Инициализация из pandas DataFrame"))
except Exception as e:
    test_results.append(("❌ FAILED", f"Инициализация из pandas DataFrame - {str(e)}"))

try:
    # Из dict
    ds_dict = SparkDataset(data={'data': {'id': [1, 2], 'val': [10, 20]}}, session=sp_s)
    assert ds_dict.shape[0] == 2, "Dict инициализация failed"
    test_results.append(("✅ PASSED", "Инициализация из dict"))
except Exception as e:
    test_results.append(("❌ FAILED", f"Инициализация из dict - {str(e)}"))

try:
    # Из SparkDF
    spark_df = sp_s.createDataFrame(test_numeric_df)
    ds_spark = SparkDataset(data=spark_df, session=sp_s)
    assert ds_spark.shape[0] == 5, "SparkDF инициализация failed"
    test_results.append(("✅ PASSED", "Инициализация из SparkDF"))
except Exception as e:
    test_results.append(("❌ FAILED", f"Инициализация из SparkDF - {str(e)}"))

try:
    # Пустой датасет
    ds_empty = SparkDataset(data=None, session=sp_s)
    assert ds_empty.data is not None, "Empty инициализация failed"
    test_results.append(("✅ PASSED", "Инициализация пустого Dataset"))
except Exception as e:
    test_results.append(("❌ FAILED", f"Инициализация пустого Dataset - {str(e)}"))

for status, msg in test_results:
    print(f"{status}: {msg}")

ТЕСТ: Инициализация SparkDataset
✅ PASSED: Инициализация из pandas DataFrame
✅ PASSED: Инициализация из dict
✅ PASSED: Инициализация из SparkDF
✅ PASSED: Инициализация пустого Dataset


In [4]:
# === ТЕСТ: Операторы сравнения ===
print("=" * 60)
print("ТЕСТ: Операторы сравнения (числовые колонки)")
print("=" * 60)

test_results = []
ds = ds_numeric  # Используем числовой датасет

try:
    ds_eq = ds == 10
    assert ds_eq.data is not None, "__eq__ failed"
    test_results.append(("✅ PASSED", "__eq__ (равно)"))
except Exception as e:
    test_results.append(("❌ FAILED", f"__eq__ - {str(e)}"))

try:
    ds_ne = ds != 10
    assert ds_ne.data is not None, "__ne__ failed"
    test_results.append(("✅ PASSED", "__ne__ (не равно)"))
except Exception as e:
    test_results.append(("❌ FAILED", f"__ne__ - {str(e)}"))

try:
    ds_lt = ds < 30
    assert ds_lt.data is not None, "__lt__ failed"
    test_results.append(("✅ PASSED", "__lt__ (меньше)"))
except Exception as e:
    test_results.append(("❌ FAILED", f"__lt__ - {str(e)}"))

try:
    ds_le = ds <= 30
    assert ds_le.data is not None, "__le__ failed"
    test_results.append(("✅ PASSED", "__le__ (меньше или равно)"))
except Exception as e:
    test_results.append(("❌ FAILED", f"__le__ - {str(e)}"))

try:
    ds_ge = ds >= 30
    assert ds_ge.data is not None, "__ge__ failed"
    test_results.append(("✅ PASSED", "__ge__ (больше или равно)"))
except Exception as e:
    test_results.append(("❌ FAILED", f"__ge__ - {str(e)}"))

try:
    ds_gt = ds > 30
    assert ds_gt.data is not None, "__gt__ failed"
    test_results.append(("✅ PASSED", "__gt__ (больше)"))
except Exception as e:
    test_results.append(("❌ FAILED", f"__gt__ - {str(e)}"))

for status, msg in test_results:
    print(f"{status}: {msg}")

ТЕСТ: Операторы сравнения (числовые колонки)
✅ PASSED: __eq__ (равно)
✅ PASSED: __ne__ (не равно)
✅ PASSED: __lt__ (меньше)
✅ PASSED: __le__ (меньше или равно)
✅ PASSED: __ge__ (больше или равно)
✅ PASSED: __gt__ (больше)


In [5]:
# === ТЕСТ: Унарные операторы ===
print("=" * 60)
print("ТЕСТ: Унарные операторы (числовые колонки)")
print("=" * 60)

test_results = []
ds = ds_numeric

try:
    ds_pos = +ds
    assert ds_pos.data is not None, "__pos__ failed"
    test_results.append(("✅ PASSED", "__pos__ (плюс унарный)"))
except Exception as e:
    test_results.append(("❌ FAILED", f"__pos__ - {str(e)}"))

try:
    ds_neg = -ds
    assert ds_neg.data is not None, "__neg__ failed"
    test_results.append(("✅ PASSED", "__neg__ (минус унарный)"))
except Exception as e:
    test_results.append(("❌ FAILED", f"__neg__ - {str(e)}"))

try:
    ds_abs = abs(ds)
    assert ds_abs.data is not None, "__abs__ failed"
    test_results.append(("✅ PASSED", "__abs__ (абсолютное значение)"))
except Exception as e:
    test_results.append(("❌ FAILED", f"__abs__ - {str(e)}"))

try:
    ds_round = round(ds, 2)
    assert ds_round.data is not None, "__round__ failed"
    test_results.append(("✅ PASSED", "__round__ (округление)"))
except Exception as e:
    test_results.append(("❌ FAILED", f"__round__ - {str(e)}"))

# __invert__ работает только с boolean колонками
try:
    ds_bool = ds_numeric > 20  # Создаем boolean датасет
    ds_inv = ~ds_bool
    assert ds_inv.data is not None, "__invert__ failed"
    test_results.append(("✅ PASSED", "__invert__ (инверсия boolean)"))
except Exception as e:
    test_results.append(("❌ FAILED", f"__invert__ - {str(e)}"))

for status, msg in test_results:
    print(f"{status}: {msg}")

ТЕСТ: Унарные операторы (числовые колонки)
✅ PASSED: __pos__ (плюс унарный)
✅ PASSED: __neg__ (минус унарный)
✅ PASSED: __abs__ (абсолютное значение)
✅ PASSED: __round__ (округление)
✅ PASSED: __invert__ (инверсия boolean)


In [6]:
# === ТЕСТ: Арифметические операторы (бинарные) ===
print("=" * 60)
print("ТЕСТ: Арифметические операторы (бинарные)")
print("=" * 60)

test_results = []
ds = ds_numeric

try:
    ds_add = ds + 10
    assert ds_add.data is not None, "__add__ failed"
    test_results.append(("✅ PASSED", "__add__ (сложение)"))
except Exception as e:
    test_results.append(("❌ FAILED", f"__add__ - {str(e)}"))

try:
    ds_sub = ds - 5
    assert ds_sub.data is not None, "__sub__ failed"
    test_results.append(("✅ PASSED", "__sub__ (вычитание)"))
except Exception as e:
    test_results.append(("❌ FAILED", f"__sub__ - {str(e)}"))

try:
    ds_mul = ds * 2
    assert ds_mul.data is not None, "__mul__ failed"
    test_results.append(("✅ PASSED", "__mul__ (умножение)"))
except Exception as e:
    test_results.append(("❌ FAILED", f"__mul__ - {str(e)}"))

try:
    ds_fdiv = ds // 2
    assert ds_fdiv.data is not None, "__floordiv__ failed"
    test_results.append(("✅ PASSED", "__floordiv__ (целочисленное деление)"))
except Exception as e:
    test_results.append(("❌ FAILED", f"__floordiv__ - {str(e)}"))

try:
    ds_tdiv = ds / 2
    assert ds_tdiv.data is not None, "__truediv__ failed"
    test_results.append(("✅ PASSED", "__truediv__ (деление)"))
except Exception as e:
    test_results.append(("❌ FAILED", f"__truediv__ - {str(e)}"))

try:
    ds_mod = ds % 3
    assert ds_mod.data is not None, "__mod__ failed"
    test_results.append(("✅ PASSED", "__mod__ (остаток от деления)"))
except Exception as e:
    test_results.append(("❌ FAILED", f"__mod__ - {str(e)}"))

try:
    ds_pow = ds ** 2
    assert ds_pow.data is not None, "__pow__ failed"
    test_results.append(("✅ PASSED", "__pow__ (возведение в степень)"))
except Exception as e:
    test_results.append(("❌ FAILED", f"__pow__ - {str(e)}"))

for status, msg in test_results:
    print(f"{status}: {msg}")

ТЕСТ: Арифметические операторы (бинарные)
✅ PASSED: __add__ (сложение)
✅ PASSED: __sub__ (вычитание)
✅ PASSED: __mul__ (умножение)
✅ PASSED: __floordiv__ (целочисленное деление)
✅ PASSED: __truediv__ (деление)
✅ PASSED: __mod__ (остаток от деления)
✅ PASSED: __pow__ (возведение в степень)


In [7]:
# === ТЕСТ: Обратные арифметические операторы ===
print("=" * 60)
print("ТЕСТ: Обратные арифметические операторы")
print("=" * 60)

test_results = []
ds = ds_numeric

try:
    ds_radd = 10 + ds
    assert ds_radd.data is not None, "__radd__ failed"
    test_results.append(("✅ PASSED", "__radd__ (обратное сложение)"))
except Exception as e:
    test_results.append(("❌ FAILED", f"__radd__ - {str(e)}"))

try:
    ds_rsub = 100 - ds
    assert ds_rsub.data is not None, "__rsub__ failed"
    test_results.append(("✅ PASSED", "__rsub__ (обратное вычитание)"))
except Exception as e:
    test_results.append(("❌ FAILED", f"__rsub__ - {str(e)}"))

try:
    ds_rmul = 3 * ds
    assert ds_rmul.data is not None, "__rmul__ failed"
    test_results.append(("✅ PASSED", "__rmul__ (обратное умножение)"))
except Exception as e:
    test_results.append(("❌ FAILED", f"__rmul__ - {str(e)}"))

try:
    ds_rfdiv = 100 // ds
    assert ds_rfdiv.data is not None, "__rfloordiv__ failed"
    test_results.append(("✅ PASSED", "__rfloordiv__ (обратное целочисленное деление)"))
except Exception as e:
    test_results.append(("❌ FAILED", f"__rfloordiv__ - {str(e)}"))

try:
    ds_rtdiv = 100 / ds
    assert ds_rtdiv.data is not None, "__rtruediv__ failed"
    test_results.append(("✅ PASSED", "__rtruediv__ (обратное деление)"))
except Exception as e:
    test_results.append(("❌ FAILED", f"__rtruediv__ - {str(e)}"))

try:
    ds_rmod = 100 % ds
    assert ds_rmod.data is not None, "__rmod__ failed"
    test_results.append(("✅ PASSED", "__rmod__ (обратный остаток)"))
except Exception as e:
    test_results.append(("❌ FAILED", f"__rmod__ - {str(e)}"))

try:
    ds_rpow = 2 ** ds
    assert ds_rpow.data is not None, "__rpow__ failed"
    test_results.append(("✅ PASSED", "__rpow__ (обратная степень)"))
except Exception as e:
    test_results.append(("❌ FAILED", f"__rpow__ - {str(e)}"))

for status, msg in test_results:
    print(f"{status}: {msg}")

ТЕСТ: Обратные арифметические операторы
✅ PASSED: __radd__ (обратное сложение)
✅ PASSED: __rsub__ (обратное вычитание)
✅ PASSED: __rmul__ (обратное умножение)
✅ PASSED: __rfloordiv__ (обратное целочисленное деление)
✅ PASSED: __rtruediv__ (обратное деление)
✅ PASSED: __rmod__ (обратный остаток)
✅ PASSED: __rpow__ (обратная степень)


In [8]:
# === ТЕСТ: Логические операторы между SparkDataset ===
print("=" * 60)
print("ТЕСТ: Логические операторы между SparkDataset")
print("=" * 60)

test_results = []

# Создаем два boolean датасета с одинаковой структурой
bool_data1 = {
    'flag1': [True, False, True, False, True],
    'flag2': [False, True, True, False, False],
}
bool_data2 = {
    'flag1': [True, True, False, False, True],
    'flag2': [True, False, True, False, True],
}

df_bool1 = pd.DataFrame(bool_data1)
df_bool2 = pd.DataFrame(bool_data2)

ds_bool1 = SparkDataset(data=df_bool1, session=sp_s)
ds_bool2 = SparkDataset(data=df_bool2, session=sp_s)

# Тест __and__ между двумя SparkDataset
try:
    ds_and = ds_bool1 & ds_bool2
    result = ds_and.data.collect()
    assert len(result) == 5, "__and__ failed - wrong row count"
    # Проверка значений: flag1: [T, F, F, F, T], flag2: [F, F, T, F, F]
    test_results.append(("✅ PASSED", "__and__ между двумя SparkDataset"))
except Exception as e:
    test_results.append(("❌ FAILED", f"__and__ между SparkDataset - {str(e)}"))

# Тест __or__ между двумя SparkDataset
try:
    ds_or = ds_bool1 | ds_bool2
    result = ds_or.data.collect()
    assert len(result) == 5, "__or__ failed - wrong row count"
    test_results.append(("✅ PASSED", "__or__ между двумя SparkDataset"))
except Exception as e:
    test_results.append(("❌ FAILED", f"__or__ между SparkDataset - {str(e)}"))

# Тест __xor__ между двумя SparkDataset
try:
    ds_xor = ds_bool1 ^ ds_bool2
    result = ds_xor.data.collect()
    assert len(result) == 5, "__xor__ failed - wrong row count"
    test_results.append(("✅ PASSED", "__xor__ между двумя SparkDataset"))
except Exception as e:
    test_results.append(("❌ FAILED", f"__xor__ между SparkDataset - {str(e)}"))

# Тест с scalar (должно работать и после исправления)
try:
    ds_and_scalar = ds_bool1 & True
    assert ds_and_scalar.data is not None
    test_results.append(("✅ PASSED", "__and__ с scalar (True)"))
except Exception as e:
    test_results.append(("❌ FAILED", f"__and__ с scalar - {str(e)}"))

try:
    ds_or_scalar = ds_bool1 | False
    assert ds_or_scalar.data is not None
    test_results.append(("✅ PASSED", "__or__ с scalar (False)"))
except Exception as e:
    test_results.append(("❌ FAILED", f"__or__ с scalar - {str(e)}"))

# Тест __invert__
try:
    ds_inv = ~ds_bool1
    assert ds_inv.data is not None
    test_results.append(("✅ PASSED", "__invert__ (инверсия)"))
except Exception as e:
    test_results.append(("❌ FAILED", f"__invert__ - {str(e)}"))

for status, msg in test_results:
    print(f"{status}: {msg}")

print(f"\nИтого: {sum(1 for s, _ in test_results if '✅' in s)} passed, {sum(1 for s, _ in test_results if '❌' in s)} failed")

ТЕСТ: Логические операторы между SparkDataset
✅ PASSED: __and__ между двумя SparkDataset
✅ PASSED: __or__ между двумя SparkDataset
✅ PASSED: __xor__ между двумя SparkDataset
✅ PASSED: __and__ с scalar (True)
✅ PASSED: __or__ с scalar (False)
✅ PASSED: __invert__ (инверсия)

Итого: 6 passed, 0 failed


In [9]:
# === ТЕСТ: Методы представления и навигации ===
print("=" * 60)
print("ТЕСТ: Методы представления и навигации")
print("=" * 60)

test_results = []
ds = ds_mixed

try:
    repr_str = repr(ds)
    assert 'SparkNavigation' in repr_str, "__repr__ failed"
    test_results.append(("✅ PASSED", f"__repr__ - {repr_str[:60]}..."))
except Exception as e:
    test_results.append(("❌ FAILED", f"__repr__ - {str(e)}"))

try:
    cols = ds.columns
    assert len(cols) == 5, f"columns failed: {len(cols)}"
    test_results.append(("✅ PASSED", f"columns - {cols}"))
except Exception as e:
    test_results.append(("❌ FAILED", f"columns - {str(e)}"))

try:
    shape = ds.shape
    assert shape[0] == 5 and shape[1] == 5, f"shape failed: {shape}"
    test_results.append(("✅ PASSED", f"shape - {shape}"))
except Exception as e:
    test_results.append(("❌ FAILED", f"shape - {str(e)}"))

try:
    ds_get = ds.get(['id', 'name'])
    assert len(ds_get.columns) == 2, "get failed"
    test_results.append(("✅ PASSED", "get (выбор колонок)"))
except Exception as e:
    test_results.append(("❌ FAILED", f"get - {str(e)}"))

try:
    ds_take = ds.take([0, 1], axis=1)
    assert len(ds_take.columns) == 2, "take failed"
    test_results.append(("✅ PASSED", "take (выбор колонок по индексу)"))
except Exception as e:
    test_results.append(("❌ FAILED", f"take - {str(e)}"))

try:
    ds_empty = ds.create_empty(columns=['a', 'b', 'c'])
    assert len(ds_empty.columns) == 3, "create_empty failed"
    test_results.append(("✅ PASSED", "create_empty"))
except Exception as e:
    test_results.append(("❌ FAILED", f"create_empty - {str(e)}"))

for status, msg in test_results:
    print(f"{status}: {msg}")

ТЕСТ: Методы представления и навигации
✅ PASSED: __repr__ - SparkNavigation(rows=5, columns=5, schema=[id:bigint, name:s...
✅ PASSED: columns - ['id', 'name', 'value', 'category', 'active']
✅ PASSED: shape - (5, 5)
✅ PASSED: get (выбор колонок)
✅ PASSED: take (выбор колонок по индексу)
✅ PASSED: create_empty


In [10]:
# === ТЕСТ: Агрегационные методы ===
print("=" * 60)
print("ТЕСТ: Агрегационные методы")
print("=" * 60)

test_results = []
ds = ds_numeric

try:
    ds_agg = ds.agg('sum')
    assert ds_agg is not None, "agg failed"
    test_results.append(("✅ PASSED", "agg"))
except Exception as e:
    test_results.append(("❌ FAILED", f"agg - {str(e)}"))

try:
    ds_max = ds.max()
    assert ds_max is not None, "max failed"
    test_results.append(("✅ PASSED", "max"))
except Exception as e:
    test_results.append(("❌ FAILED", f"max - {str(e)}"))

try:
    ds_min = ds.min()
    assert ds_min is not None, "min failed"
    test_results.append(("✅ PASSED", "min"))
except Exception as e:
    test_results.append(("❌ FAILED", f"min - {str(e)}"))

try:
    ds_count = ds.count()
    assert ds_count is not None, "count failed"
    test_results.append(("✅ PASSED", f"count - {ds_count}"))
except Exception as e:
    test_results.append(("❌ FAILED", f"count - {str(e)}"))

try:
    ds_sum = ds.sum()
    assert ds_sum is not None, "sum failed"
    test_results.append(("✅ PASSED", "sum"))
except Exception as e:
    test_results.append(("❌ FAILED", f"sum - {str(e)}"))

try:
    ds_mean = ds.mean()
    assert ds_mean is not None, "mean failed"
    test_results.append(("✅ PASSED", "mean"))
except Exception as e:
    test_results.append(("❌ FAILED", f"mean - {str(e)}"))

try:
    ds_var = ds.var()
    assert ds_var is not None, "var failed"
    test_results.append(("✅ PASSED", "var"))
except Exception as e:
    test_results.append(("❌ FAILED", f"var - {str(e)}"))

try:
    ds_std = ds.std()
    assert ds_std is not None, "std failed"
    test_results.append(("✅ PASSED", "std"))
except Exception as e:
    test_results.append(("❌ FAILED", f"std - {str(e)}"))

for status, msg in test_results:
    print(f"{status}: {msg}")

ТЕСТ: Агрегационные методы
✅ PASSED: agg
✅ PASSED: max
✅ PASSED: min
✅ PASSED: count - SparkNavigation(rows=1, columns=4, schema=[id:bigint, value1:bigint, value2:bigint, value3:bigint, ⏣index:bigint, ... (6 total)])
✅ PASSED: sum
✅ PASSED: mean
✅ PASSED: var
✅ PASSED: std


In [11]:
# === ТЕСТ: Методы работы с пропусками (ИСПРАВЛЕННЫЕ) ===
print("=" * 60)
print("ТЕСТ: Методы работы с пропусками (ИСПРАВЛЕННЫЕ)")
print("=" * 60)

test_results = []

# Создаем датасет с разными типами данных
test_mixed_data = {
    'id': [1, 2, 3, 4, 5],
    'name': ['A', 'B', 'C', 'D', 'E'],
    'value': [10.5, 20.3, 30.1, 40.7, 50.9],
    'category': ['X', 'Y', 'X', 'Y', 'Z'],
    'active': [True, False, True, True, False]
}
test_mixed_df = pd.DataFrame(test_mixed_data)
ds_mixed = SparkDataset(data=test_mixed_df, session=sp_s)

# Тест isna
try:
    ds_isna = ds_mixed.isna()
    assert ds_isna.data is not None, "isna failed"
    
    # ✅ ПРОВЕРЯЕМ ТОЛЬКО ПУБЛИЧНЫЕ КОЛОНКИ
    schema_types = {
        f.name: f.dataType.typeName() 
        for f in ds_isna.data.schema.fields 
        if f.name in ds_isna._public_columns  # ← Исключаем служебные колонки
    }
    
    all_boolean = all(t == 'boolean' for t in schema_types.values())
    assert all_boolean, f"isna result should be all boolean, got: {schema_types}"
    test_results.append(("✅ PASSED", "isna (смешанные типы)"))
except Exception as e:
    test_results.append(("❌ FAILED", f"isna - {str(e)}"))

# Тест na_counts
try:
    na_cnt = ds_mixed.na_counts()
    assert na_cnt is not None, "na_counts failed"
    test_results.append(("✅ PASSED", f"na_counts - {na_cnt}"))
except Exception as e:
    test_results.append(("❌ FAILED", f"na_counts - {str(e)}"))

# Тест с реальными NULL значениями
test_null_data = {
    'id': [1, 2, None, 4, 5],
    'value': [10.5, None, 30.1, 40.7, 50.9],
    'name': ['A', 'B', 'C', None, 'E']
}
test_null_df = pd.DataFrame(test_null_data)
ds_null = SparkDataset(data=test_null_df, session=sp_s)

try:
    ds_null_isna = ds_null.isna()
    result = ds_null_isna.data.collect()
    # Проверяем что NULL значения обнаружены
    test_results.append(("✅ PASSED", "isna (с NULL значениями)"))
except Exception as e:
    test_results.append(("❌ FAILED", f"isna с NULL - {str(e)}"))

try:
    ds_null_na_counts = ds_null.na_counts()
    test_results.append(("✅ PASSED", f"na_counts (с NULL) - {ds_null_na_counts}"))
except Exception as e:
    test_results.append(("❌ FAILED", f"na_counts с NULL - {str(e)}"))

# Тест dropna
try:
    ds_dropna = ds_null.dropna()
    assert ds_dropna.data is not None, "dropna failed"
    test_results.append(("✅ PASSED", f"dropna - {ds_dropna.shape[0]} строк"))
except Exception as e:
    test_results.append(("❌ FAILED", f"dropna - {str(e)}"))

for status, msg in test_results:
    print(f"{status}: {msg}")

print(f"\nИтого: {sum(1 for s, _ in test_results if '✅' in s)} passed, {sum(1 for s, _ in test_results if '❌' in s)} failed")

ТЕСТ: Методы работы с пропусками (ИСПРАВЛЕННЫЕ)
✅ PASSED: isna (смешанные типы)
✅ PASSED: na_counts - {'id': 0, 'name': 0, 'value': 0, 'category': 0, 'active': 0, '⏣index': 10, '⏣_physical_index': -5}
✅ PASSED: isna (с NULL значениями)
✅ PASSED: na_counts (с NULL) - {'id': 1, 'value': 1, 'name': 1, '⏣index': 10, '⏣_physical_index': -5}
✅ PASSED: dropna - 2 строк

Итого: 5 passed, 0 failed


In [12]:
# === ТЕСТ: Детальная проверка isna() и na_counts() ===
print("=" * 60)
print("ТЕСТ: Детальная проверка isna() и na_counts()")
print("=" * 60)

# Создаем датасет с NULL в разных типах
test_detail_data = {
    'int_col': [1, 2, None, 4, 5],
    'float_col': [1.1, None, 3.3, 4.4, 5.5],
    'string_col': ['A', 'B', None, 'D', 'E'],
    'bool_col': [True, False, None, True, False],
}
test_detail_df = pd.DataFrame(test_detail_data)
ds_detail = SparkDataset(data=test_detail_df, session=sp_s)

print("Исходные данные:")
ds_detail.data.show()

print("\nРезультат isna():")
ds_isna_detail = ds_detail.isna()
ds_isna_detail.data.show()

# Проверяем что NULL правильно обнаружены
result = ds_isna_detail.data.collect()
print("\nПроверка обнаружения NULL:")
for i, row in enumerate(result):
    null_count = sum([row[c] for c in ds_isna_detail.columns])
    print(f"  Строка {i}: {null_count} NULL значений")

# Проверяем na_counts
print(f"\nna_counts(): {ds_detail.na_counts()}")

# Ожидаем: int_col=1, float_col=1, string_col=1, bool_col=1
expected_nulls = {'int_col': 1, 'float_col': 1, 'string_col': 1, 'bool_col': 1}
actual_nulls = ds_detail.na_counts()

print("\nСравнение с ожидаемым:")
all_correct = True
for col, expected in expected_nulls.items():
    actual = actual_nulls.get(col, 0)
    status = "✅" if actual == expected else "❌"
    if actual != expected:
        all_correct = False
    print(f"  {status} {col}: expected={expected}, actual={actual}")

if all_correct:
    print("\n✅ Детальная проверка завершена - ВСЕ ТЕСТЫ ПРОЙДЕНЫ")
else:
    print("\n⚠️  Детальная проверка завершена - ЕСТЬ ОШИБКИ")

ТЕСТ: Детальная проверка isna() и na_counts()
Исходные данные:
+-------+---------+----------+--------+------+----------------+
|int_col|float_col|string_col|bool_col|⏣index|⏣_physical_index|
+-------+---------+----------+--------+------+----------------+
|    1.0|      1.1|         A|    true|     0|               0|
|    2.0|      NaN|         B|   false|     1|               1|
|    NaN|      3.3|      NULL|    NULL|     2|               2|
|    4.0|      4.4|         D|    true|     3|               3|
|    5.0|      5.5|         E|   false|     4|               4|
+-------+---------+----------+--------+------+----------------+


Результат isna():
+-------+---------+----------+--------+------+----------------+
|int_col|float_col|string_col|bool_col|⏣index|⏣_physical_index|
+-------+---------+----------+--------+------+----------------+
|  false|    false|     false|   false|     0|              -1|
|  false|     true|     false|   false|     1|              -1|
|   true|    false|  

In [13]:
# === ТЕСТ: Метод replace() (ИСПРАВЛЕННЫЙ) ===
print("=" * 60)
print("ТЕСТ: Метод replace() (ИСПРАВЛЕННЫЙ)")
print("=" * 60)

test_results = []

# Создаем датасет с разными типами данных
test_mixed_data = {
    'id': [1, 2, 3, 4, 5],
    'name': ['A', 'B', 'C', 'D', 'E'],
    'value': [10.5, 20.3, 30.1, 40.7, 50.9],
    'category': ['X', 'Y', 'X', 'Y', 'Z'],
    'active': [True, False, True, True, False]
}
test_mixed_df = pd.DataFrame(test_mixed_data)
ds_mixed = SparkDataset(data=test_mixed_df, session=sp_s)

# Тест replace на STRING колонке
try:
    ds_replaced = ds_mixed.replace(to_replace='X', value='A')
    result = ds_replaced.data.collect()
    # Проверяем что замена произошла
    categories = [row['category'] for row in result]
    assert 'A' in categories, "replace failed - no replacement occurred"
    test_results.append(("✅ PASSED", "replace (STRING колонка)"))
except Exception as e:
    test_results.append(("❌ FAILED", f"replace STRING - {str(e)}"))

# Тест replace на числовой колонке
try:
    ds_replaced_num = ds_mixed.replace(to_replace=1, value=100)
    result = ds_replaced_num.data.collect()
    ids = [row['id'] for row in result]
    assert 100 in ids, "replace failed - no numeric replacement"
    test_results.append(("✅ PASSED", "replace (INT колонка)"))
except Exception as e:
    test_results.append(("❌ FAILED", f"replace INT - {str(e)}"))

# Тест replace с dict
try:
    ds_replaced_dict = ds_mixed.replace({'X': 'A', 'Y': 'B'})
    result = ds_replaced_dict.data.collect()
    categories = [row['category'] for row in result]
    assert 'A' in categories and 'B' in categories, "replace dict failed"
    test_results.append(("✅ PASSED", "replace (dict)"))
except Exception as e:
    test_results.append(("❌ FAILED", f"replace dict - {str(e)}"))

# Тест replace с subset (только определённые колонки)
try:
    ds_replaced_subset = ds_mixed.replace(to_replace='X', value='A', subset=['category'])
    result = ds_replaced_subset.data.collect()
    test_results.append(("✅ PASSED", "replace (subset)"))
except Exception as e:
    test_results.append(("❌ FAILED", f"replace subset - {str(e)}"))

# Тест replace на BOOLEAN колонке
try:
    ds_bool_data = {'flag': [True, False, True, False]}
    ds_bool = SparkDataset(data=pd.DataFrame(ds_bool_data), session=sp_s)
    ds_bool_replaced = ds_bool.replace(to_replace=True, value=False)
    result = ds_bool_replaced.data.collect()
    test_results.append(("✅ PASSED", "replace (BOOLEAN колонка)"))
except Exception as e:
    test_results.append(("❌ FAILED", f"replace BOOLEAN - {str(e)}"))

for status, msg in test_results:
    print(f"{status}: {msg}")

print(f"\nИтого: {sum(1 for s, _ in test_results if '✅' in s)} passed, {sum(1 for s, _ in test_results if '❌' in s)} failed")

ТЕСТ: Метод replace() (ИСПРАВЛЕННЫЙ)
✅ PASSED: replace (STRING колонка)
✅ PASSED: replace (INT колонка)
✅ PASSED: replace (dict)
✅ PASSED: replace (subset)
✅ PASSED: replace (BOOLEAN колонка)

Итого: 5 passed, 0 failed


In [14]:
# === ТЕСТ: Инициализация SparkDataset ===
print("=" * 60)
print("ТЕСТ: Инициализация SparkDataset")
print("=" * 60)

try:
    # Создаем тестовый DataFrame
    test_data = {
        'id': [1, 2, 3, 4, 5],
        'name': ['A', 'B', 'C', 'D', 'E'],
        'value': [10.5, 20.3, 30.1, 40.7, 50.9],
        'category': ['X', 'Y', 'X', 'Y', 'Z'],
        'active': [True, False, True, True, False]
    }
    test_df = pd.DataFrame(test_data)
    
    # Создаем SparkDataset
    ds = SparkDataset(data=test_df, session=sp_s)
    
    assert ds.data is not None, "Data не инициализирован"
    assert len(ds.columns) == 5, f"Ожидалось 5 колонок, получено {len(ds.columns)}"
    assert ds.shape[0] == 5, f"Ожидалось 5 строк, получено {ds.shape[0]}"
    
    print("✅ PASSED: Инициализация SparkDataset")
except Exception as e:
    print(f"❌ FAILED: Инициализация SparkDataset - {str(e)}")

# === ТЕСТ: Инициализация из разных типов данных ===
print("\n" + "=" * 60)
print("ТЕСТ: Инициализация из разных типов данных")
print("=" * 60)

try:
    # Из dict
    ds_dict = SparkDataset(data={'data': {'id': [1, 2], 'val': [10, 20]}}, session=sp_s)
    assert ds_dict.shape[0] == 2, "Dict инициализация failed"
    print("✅ PASSED: Инициализация из dict")
    
    # Из SparkDF
    spark_df = sp_s.createDataFrame(test_df)
    ds_spark = SparkDataset(data=spark_df, session=sp_s)
    assert ds_spark.shape[0] == 5, "SparkDF инициализация failed"
    print("✅ PASSED: Инициализация из SparkDF")
    
    # Из None (пустой)
    ds_empty = SparkDataset(data=None, session=sp_s)
    assert ds_empty.data is not None, "Empty инициализация failed"
    print("✅ PASSED: Инициализация пустого Dataset")
    
except Exception as e:
    print(f"❌ FAILED: Инициализация из разных типов - {str(e)}")

ТЕСТ: Инициализация SparkDataset
✅ PASSED: Инициализация SparkDataset

ТЕСТ: Инициализация из разных типов данных
✅ PASSED: Инициализация из dict
✅ PASSED: Инициализация из SparkDF
✅ PASSED: Инициализация пустого Dataset


In [15]:
# === ТЕСТ: Методы сортировки и группировки ===
print("=" * 60)
print("ТЕСТ: Методы сортировки и группировки")
print("=" * 60)

try:
    ds = SparkDataset(data=test_df, session=sp_s)
    
    # sort_values
    ds_sorted = ds.sort_values(by='value', ascending=False)
    assert ds_sorted.data is not None, "sort_values failed"
    print("✅ PASSED: sort_values")
    
    # value_counts
    ds_vc = ds.select_dtypes(include=['str']).value_counts()
    assert ds_vc.data is not None, "value_counts failed"
    print("✅ PASSED: value_counts")
    
    # groupby
    groups = ds.groupby(by='category')
    assert len(groups) > 0, "groupby failed"
    print(f"✅ PASSED: groupby - {len(groups)} групп")
    
except Exception as e:
    print(f"❌ FAILED: Методы сортировки и группировки - {str(e)}")

ТЕСТ: Методы сортировки и группировки
✅ PASSED: sort_values
✅ PASSED: value_counts
✅ PASSED: groupby - 3 групп


In [16]:
# === ТЕСТ: Методы выборки и трансформации ===
print("=" * 60)
print("ТЕСТ: Методы выборки и трансформации")
print("=" * 60)

try:
    ds = SparkDataset(data=test_df, session=sp_s)
    
    # sample (frac)
    ds_sample_frac = ds.sample(frac=0.5, random_state=42)
    assert ds_sample_frac.data is not None, "sample frac failed"
    print(f"✅ PASSED: sample (frac) - {ds_sample_frac.shape[0]} строк")
    
    # sample (n)
    ds_sample_n = ds.sample(n=3, random_state=42)
    assert ds_sample_n.data is not None, "sample n failed"
    print(f"✅ PASSED: sample (n) - {ds_sample_n.shape[0]} строк")
    
    # transpose
    ds_trans = ds.select_dtypes(include=['int', 'float']).transpose()
    assert ds_trans.data is not None, "transpose failed"
    print("✅ PASSED: transpose")
    
except Exception as e:
    print(f"❌ FAILED: Методы выборки и трансформации - {str(e)}")

ТЕСТ: Методы выборки и трансформации
✅ PASSED: sample (frac) - 0 строк
✅ PASSED: sample (n) - 0 строк
✅ PASSED: transpose


In [17]:
ds.sample(n=3, random_state=42)

id,name,value,category,active,⏣index,⏣_physical_index


In [18]:
# === ДИАГНОСТИКА: Почему sample() возвращает 0 строк ===
print("=" * 60)
print("ДИАГНОСТИКА: Проблема с sample()")
print("=" * 60)

# Создаем тестовый датасет
test_data = {
    'id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'value': [10.5, 20.3, 30.1, 40.7, 50.9, 60.2, 70.8, 80.4, 90.6, 100.1]
}
test_df = pd.DataFrame(test_data)
ds = SparkDataset(data=test_df, session=sp_s)

print(f"Исходный датасет: {ds.shape[0]} строк")
print(f"Количество колонок: {ds.shape[1]}")

# Проверяем count() напрямую
print(f"\nds.data.count(): {ds.data.count()}")

# Проверяем sample с разными параметрами
print("\nТест sample():")
for frac in [0.1, 0.3, 0.5, 0.7, 0.9]:
    ds_sample = ds.sample(frac=frac, random_state=42)
    print(f"  frac={frac}: {ds_sample.shape[0]} строк")

print("\nТест sample(n):")
for n in [1, 3, 5, 7, 10]:
    ds_sample = ds.sample(n=n, random_state=42)
    print(f"  n={n}: {ds_sample.shape[0]} строк")

ДИАГНОСТИКА: Проблема с sample()
Исходный датасет: 10 строк
Количество колонок: 2

ds.data.count(): 10

Тест sample():
  frac=0.1: 3 строк
  frac=0.3: 3 строк
  frac=0.5: 4 строк
  frac=0.7: 6 строк
  frac=0.9: 9 строк

Тест sample(n):
  n=1: 1 строк
  n=3: 3 строк
  n=5: 4 строк
  n=7: 6 строк
  n=10: 10 строк
